# 07. Compare adapters, reload and evaluate an untouched test set

Reload each completed non-smoke run from its saved adapters. Compare generated associations on
the same validation pages, alongside a frozen baseline with identical image settings. Report
bubble accuracy, exact-page accuracy and valid-output rate. Missing/invalid predictions are
not repaired with ground truth. Failed runs and smoke runs are excluded.

Select by validation results. If adapters do not beat the frozen baseline, keep the base model.
Evaluate only the selected model and its frozen counterpart on test; never choose among runs
using test results. Only save a small adapter bundle, not another multi-GB base model.


In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "models.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "models.json").exists():
    raise FileNotFoundError("Start the kernel in 1pageTest or its 6_lora_training folder.")
sys.path.insert(0, str(ROOT / "6_lora_training"))
import lora_helpers as H
print("Interpreter:", sys.executable)
print("Dataset:", H.PAGE_DIR)
print("Local base model:", H.MODEL_DIR)


In [ ]:
pages = H.read_pages()
val_pages = H.partition(pages, "val")
comparison = H.compare_validation(val_pages)
display(comparison)
best_adapter = comparison[0]
best_frozen = max(comparison, key=lambda row: row["frozen_bubble_accuracy"])
USE_ADAPTER = best_adapter["bubble_accuracy"] > best_frozen["frozen_bubble_accuracy"]
best = best_adapter if USE_ADAPTER else best_frozen
choice = {"run": best["name"], "use_adapter": USE_ADAPTER,
          "selection_split": "val", "validation": best}
H.write_json(H.RUN_DIR / "selection.json", choice)
print("Selected:", best["name"] if USE_ADAPTER else "frozen base (no demonstrated adapter gain)")


In [ ]:
# This cell requires separately reviewed test pages and an explicit final selection above.
test_pages = H.partition(pages, "test")
checkpoint = H.RUN_DIR / best["name"] / "best"
config = json.loads((checkpoint / "experiment.json").read_text())
model, tokenizer = H.load_base()
try:
    frozen_test = H.evaluate_model(model, tokenizer, test_pages, config, "frozen_selected_settings",
                                  H.RUN_DIR / "test_frozen.json")
finally:
    del model, tokenizer
    H.cleanup()
if USE_ADAPTER:
    adapted_test = H.evaluate_run(checkpoint, test_pages,
                                  destination=H.RUN_DIR / "test_selected_adapter.json")
    display([frozen_test["summary"], adapted_test["summary"]])
else:
    display(frozen_test["summary"])


In [ ]:
import shutil
EXPORT = ROOT / "data/training/export/selected_adapter"
if USE_ADAPTER:
    if EXPORT.exists():
        raise FileExistsError("Choose a new export folder to preserve the existing selection.")
    shutil.copytree(checkpoint, EXPORT)
    H.write_json(EXPORT / "selection.json", choice)
    print("Adapter bundle:", EXPORT)
else:
    print("No adapter exported; use the unchanged local base.")


To use the selected adapter in another notebook, load the same local base with H.load_base(),
then call H.restore_adapters(model, EXPORT). Keep the returned config's image view/max_tiles
and H.question(page) consistent. Use H.evaluate_model for inference on reviewed pages.
The original association notebook uses a different prompt/image layout and has not been
rewritten automatically. Do not merge adapters into the base checkpoint during these tests.
